# Projeto RAG simples com Qdrant local + Maritaca `sabiazinho-4`

Este notebook implementa um **RAG simples** usando o arquivo `imdb_top_1000.csv`.

Fluxo do projeto:

1. Ler o CSV de filmes.
2. Transformar cada filme em um texto.
3. Gerar embeddings leves com `sentence-transformers/all-MiniLM-L6-v2`.
4. Salvar os vetores no Qdrant local.
5. Receber perguntas do usuário.
6. Buscar os filmes mais parecidos no Qdrant.
7. Enviar pergunta + contexto para a Maritaca usando o modelo `sabiazinho-4`.
8. Repetir perguntas em um loop até o usuário digitar `sair`.

## Célula 1 — Instalação

Execute esta célula apenas se ainda não tiver instalado as dependências.

No terminal, dentro da pasta deste projeto, você também pode rodar:

```bash
pip install -r requirements.txt
```


In [ ]:
# A linha abaixo instala todas as bibliotecas listadas no arquivo requirements.txt.
# O caractere "!" permite executar comandos de terminal dentro do Jupyter Notebook.
# Se você já instalou tudo pelo terminal, pode pular esta célula.
!pip install -r requirements.txt

## Célula 2 — Imports

Aqui importamos todas as bibliotecas usadas no projeto.


In [ ]:
# Importa a biblioteca os para ler variáveis de ambiente, como a chave da API.
import os

# Importa a biblioteca time para medir tempo e fazer pequenas pausas, se necessário.
import time

# Importa Path para trabalhar com caminhos de arquivos de forma mais segura.
from pathlib import Path

# Importa pandas para ler e manipular o arquivo CSV.
import pandas as pd

# Importa numpy para lidar com vetores numéricos, como embeddings.
import numpy as np

# Importa load_dotenv para carregar a chave da API a partir do arquivo .env.
from dotenv import load_dotenv

# Importa tqdm para mostrar barra de progresso quando estivermos salvando dados no Qdrant.
from tqdm.auto import tqdm

# Importa SentenceTransformer para carregar um modelo leve de embeddings.
from sentence_transformers import SentenceTransformer

# Importa o cliente principal do Qdrant para conversar com o banco vetorial local.
from qdrant_client import QdrantClient

# Importa os modelos auxiliares do Qdrant, usados para configurar coleção e pontos.
from qdrant_client import models

# Importa a biblioteca OpenAI porque a API da Maritaca é compatível com esse formato de cliente.
from openai import OpenAI


## Célula 3 — Configurações principais

Altere aqui somente se precisar mudar o caminho do CSV, o nome da coleção ou os modelos.


In [ ]:
# Define o caminho do arquivo CSV no computador do professor.
# O prefixo r antes das aspas cria uma "raw string", útil em caminhos do Windows com barras invertidas.
CSV_PATH = Path(r"D:\Adriano\BDNSQL\15-05-2026\qdrant\Data\imdb_top_1000.csv")

# Define o nome da coleção dentro do Qdrant.
# Uma coleção é como uma "tabela" onde os vetores serão guardados.
COLLECTION_NAME = "imdb_top_1000_rag"

# Define o endereço do Qdrant local.
# Esse endereço é o padrão quando o Qdrant está rodando localmente na porta 6333.
QDRANT_URL = "http://localhost:6333"

# Define o modelo de embedding.
# Este modelo é leve, roda em CPU e é bom para uma aula introdutória.
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Define o modelo da Maritaca que será usado para gerar as respostas finais.
MARITACA_MODEL_NAME = "sabiazinho-4"

# Define a URL base da API da Maritaca.
# Essa URL será usada pelo cliente OpenAI compatível.
MARITACA_BASE_URL = "https://chat.maritaca.ai/api"

# Define quantos resultados parecidos serão recuperados do Qdrant para cada pergunta.
TOP_K = 5

# Define o tamanho do lote usado ao inserir dados no Qdrant.
# Lotes evitam mandar tudo de uma vez e ajudam a acompanhar o progresso.
BATCH_SIZE = 64

# Mostra as configurações principais para o aluno verificar no output do notebook.
print("Configurações carregadas:")
print(f"CSV_PATH: {CSV_PATH}")
print(f"COLLECTION_NAME: {COLLECTION_NAME}")
print(f"QDRANT_URL: {QDRANT_URL}")
print(f"EMBEDDING_MODEL_NAME: {EMBEDDING_MODEL_NAME}")
print(f"MARITACA_MODEL_NAME: {MARITACA_MODEL_NAME}")
print(f"TOP_K: {TOP_K}")
print(f"BATCH_SIZE: {BATCH_SIZE}")

## Célula 4 — Carregar `.env` e validar chave da Maritaca



In [ ]:
# Carrega as variáveis de ambiente que estão no arquivo .env.
load_dotenv()

# Lê a chave da API da Maritaca a partir da variável MARITACA_API_KEY.
MARITACA_API_KEY = os.getenv("MARITACA_API_KEY")

# Verifica se a chave foi encontrada no ambiente.
# Se a chave não existir, o notebook para aqui com uma mensagem clara.
if not MARITACA_API_KEY:
    raise ValueError(
        "Não encontrei a variável MARITACA_API_KEY. "
        "Crie um arquivo .env com: MARITACA_API_KEY=sua_chave_aqui"
    )

# Mostra uma mensagem segura, sem revelar a chave completa.
print("Chave da Maritaca encontrada no .env.")

# Mostra somente os 4 primeiros caracteres para confirmar que alguma chave foi carregada.
print(f"Início da chave carregada: {MARITACA_API_KEY[:4]}****")

## Célula 5 — Testar conexão com Qdrant local

Antes de executar esta célula, o Qdrant precisa estar rodando.

'Exemplo com Docker no terminal:

```bash
docker run -p 6333:6333 -p 6334:6334 -v "%cd%/qdrant_storage:/qdrant/storage" qdrant/qdrant
```


In [ ]:
# Cria um cliente para conversar com o Qdrant local.
qdrant_client = QdrantClient(url=QDRANT_URL)

# Pede ao Qdrant a lista de coleções existentes.
collections = qdrant_client.get_collections()

# Mostra no notebook que a conexão funcionou.
print("Conexão com o Qdrant realizada com sucesso.")

# Mostra as coleções existentes para o aluno ver o estado atual do banco.
print("Coleções encontradas no Qdrant:")
print(collections)

## Célula 6 — Ler o CSV do IMDb

Nesta etapa, vamos carregar o arquivo `imdb_top_1000.csv`.


In [ ]:
# Verifica se o caminho do CSV existe no computador.
# Se o arquivo não for encontrado, o notebook mostra uma mensagem clara.
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Arquivo CSV não encontrado em: {CSV_PATH}\n"
        "Confira se o caminho está correto."
    )

# Lê o arquivo CSV com pandas.
# O pandas transforma a planilha em uma tabela chamada DataFrame.
df = pd.read_csv(CSV_PATH)

# Mostra quantas linhas e colunas existem no arquivo.
print(f"Arquivo carregado com sucesso: {df.shape[0]} linhas e {df.shape[1]} colunas.")

# Mostra os nomes das colunas para o aluno entender a estrutura dos dados.
print("Colunas do arquivo:")
print(list(df.columns))

# Mostra as 3 primeiras linhas para uma inspeção rápida.
display(df.head(3))


## Célula 7 — Limpeza simples dos dados

Vamos remover linhas sem título ou sem descrição, porque elas não ajudam no RAG.


In [ ]:
# Cria uma cópia da tabela original para preservar o DataFrame bruto.
df_clean = df.copy()

# Remove espaços extras dos nomes das colunas.
df_clean.columns = [col.strip() for col in df_clean.columns]

# Define as colunas principais esperadas no arquivo imdb_top_1000.csv.
expected_columns = ["Series_Title", "Overview"]

# Percorre cada coluna obrigatória para validar se ela existe no CSV.
for col in expected_columns:
    # Se alguma coluna não existir, o notebook para e informa o problema.
    if col not in df_clean.columns:
        raise ValueError(f"A coluna obrigatória '{col}' não foi encontrada no CSV.")

# Remove linhas em que o título do filme está vazio.
df_clean = df_clean.dropna(subset=["Series_Title"])

# Remove linhas em que a descrição do filme está vazia.
df_clean = df_clean.dropna(subset=["Overview"])

# Troca valores ausentes por string vazia para evitar erros ao montar textos.
df_clean = df_clean.fillna("")

# Reinicia o índice da tabela para ficar de 0 até N-1.
df_clean = df_clean.reset_index(drop=True)

# Mostra o tamanho final da base após limpeza.
print(f"Base após limpeza: {df_clean.shape[0]} filmes.")

# Mostra uma amostra simples da base limpa.
display(df_clean[["Series_Title", "Overview"]].head(5))

## Célula 8 — Função para montar o texto de cada filme

O RAG precisa transformar cada linha do CSV em um texto. Esse texto será convertido em vetor.


In [ ]:
# Define uma função chamada safe_get.
# Ela pega um valor de uma linha do DataFrame sem quebrar se a coluna não existir.
def safe_get(row, column_name, default=""):
    # Verifica se a coluna existe na linha.
    if column_name in row:
        # Retorna o valor da coluna, caso ela exista.
        return row[column_name]
    # Retorna o valor padrão se a coluna não existir.
    return default


# Define uma função para transformar uma linha do CSV em um texto explicativo.
def build_movie_text(row):
    # Pega o título do filme.
    title = safe_get(row, "Series_Title")

    # Pega o ano de lançamento do filme.
    year = safe_get(row, "Released_Year")

    # Pega os gêneros do filme.
    genre = safe_get(row, "Genre")

    # Pega a nota IMDb do filme.
    rating = safe_get(row, "IMDB_Rating")

    # Pega o diretor do filme.
    director = safe_get(row, "Director")

    # Pega o ator ou atriz principal 1.
    star1 = safe_get(row, "Star1")

    # Pega o ator ou atriz principal 2.
    star2 = safe_get(row, "Star2")

    # Pega o ator ou atriz principal 3.
    star3 = safe_get(row, "Star3")

    # Pega o ator ou atriz principal 4.
    star4 = safe_get(row, "Star4")

    # Pega a descrição geral do filme.
    overview = safe_get(row, "Overview")

    # Monta uma lista com os nomes do elenco que não estão vazios.
    stars = [star for star in [star1, star2, star3, star4] if str(star).strip() != ""]

    # Junta os nomes do elenco em uma única string separada por vírgulas.
    stars_text = ", ".join(stars)

    # Monta um texto único contendo os principais campos do filme.
    # Esse texto é o que será transformado em embedding.
    text = (
        f"Título: {title}. "
        f"Ano: {year}. "
        f"Gênero: {genre}. "
        f"Nota IMDb: {rating}. "
        f"Direção: {director}. "
        f"Elenco: {stars_text}. "
        f"Sinopse: {overview}"
    )

    # Retorna o texto pronto.
    return text


# Aplica a função build_movie_text em cada linha da tabela.
df_clean["rag_text"] = df_clean.apply(build_movie_text, axis=1)

# Mostra um exemplo de texto que será enviado ao modelo de embedding.
print("Exemplo de texto que será transformado em embedding:")
print("-" * 80)
print(df_clean.loc[0, "rag_text"])
print("-" * 80)


## Célula 9 — Carregar o modelo leve de embeddings

O embedding transforma texto em uma lista de números. Esses números representam o significado do texto.


In [ ]:
# Mostra uma mensagem antes de carregar o modelo.
print("Carregando modelo de embeddings...")

# Carrega o modelo leve escolhido nas configurações.
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

# Descobre a dimensão dos vetores produzidos pelo modelo.
VECTOR_SIZE = embedding_model.get_sentence_embedding_dimension()

# Mostra a dimensão dos embeddings.
print(f"Modelo carregado: {EMBEDDING_MODEL_NAME}")

# Mostra o tamanho do vetor produzido por esse modelo.
print(f"Tamanho do vetor de embedding: {VECTOR_SIZE}")

## Célula 10 — Exemplo visual de embedding

Aqui vamos transformar um único texto em vetor para exemplo


In [ ]:
# Escolhe o primeiro texto da base como exemplo.
sample_text = df_clean.loc[0, "rag_text"]

# Gera o embedding do texto de exemplo.
sample_embedding = embedding_model.encode(sample_text)

# Converte o embedding para numpy array, caso ele ainda não esteja nesse formato.
sample_embedding = np.array(sample_embedding)

# Mostra o texto original usado no exemplo.
print("Texto de exemplo:")
print(sample_text)

# Pula uma linha para melhorar a leitura.
print()

# Mostra o formato do vetor gerado.
print(f"Formato do vetor gerado: {sample_embedding.shape}")

# Mostra apenas os 10 primeiros números do vetor para não poluir a tela.
print("Primeiros 10 números do embedding:")

# Arredonda os valores para facilitar a visualização.
print(np.round(sample_embedding[:10], 4))

## Célula 11 — Criar ou recriar a coleção no Qdrant

Esta célula cria uma coleção vetorial no Qdrant.

> Atenção: ela apaga a coleção anterior com o mesmo nome, caso exista.


In [ ]:
# Busca a lista de coleções existentes no Qdrant.
existing_collections = qdrant_client.get_collections().collections

# Extrai apenas os nomes das coleções existentes.
existing_collection_names = [collection.name for collection in existing_collections]

# Verifica se a coleção deste projeto já existe.
if COLLECTION_NAME in existing_collection_names:
    # Mostra aviso de que a coleção antiga será removida.
    print(f"A coleção '{COLLECTION_NAME}' já existe e será removida.")

    # Apaga a coleção antiga para começar a aula do zero.
    qdrant_client.delete_collection(collection_name=COLLECTION_NAME)

    # Mostra que a coleção antiga foi removida.
    print("Coleção antiga removida.")

# Cria uma nova coleção no Qdrant.
qdrant_client.create_collection(
    # Define o nome da coleção.
    collection_name=COLLECTION_NAME,

    # Define as configurações dos vetores.
    vectors_config=models.VectorParams(
        # Define o tamanho do vetor, que precisa ser igual ao tamanho do embedding.
        size=VECTOR_SIZE,

        # Define a métrica de distância.
        # COSINE é muito usada para comparar similaridade semântica entre textos.
        distance=models.Distance.COSINE,
    ),
)

# Mostra que a coleção foi criada com sucesso.
print(f"Coleção criada com sucesso: {COLLECTION_NAME}")

## Célula 12 — Função para preparar payloads do Qdrant

O payload guarda as informações textuais do filme junto com o vetor.


In [ ]:
# Define uma função para montar o payload de um filme.
def build_payload(row):
    # Cria um dicionário vazio para armazenar os metadados.
    payload = {}

    # Salva o texto completo usado no RAG.
    payload["rag_text"] = str(safe_get(row, "rag_text"))

    # Salva o título do filme.
    payload["title"] = str(safe_get(row, "Series_Title"))

    # Salva o ano de lançamento.
    payload["year"] = str(safe_get(row, "Released_Year"))

    # Salva o gênero.
    payload["genre"] = str(safe_get(row, "Genre"))

    # Salva a nota IMDb.
    payload["imdb_rating"] = str(safe_get(row, "IMDB_Rating"))

    # Salva o diretor.
    payload["director"] = str(safe_get(row, "Director"))

    # Salva a sinopse.
    payload["overview"] = str(safe_get(row, "Overview"))

    # Retorna o dicionário de metadados.
    return payload


# Mostra um exemplo de payload para o primeiro filme.
example_payload = build_payload(df_clean.iloc[0])

# Imprime o payload de exemplo.
print("Exemplo de payload que será salvo no Qdrant:")
print(example_payload)


## Célula 13 — Gerar embeddings e salvar tudo no Qdrant

Esta é a etapa principal da base vetorial.

Usamos `tqdm` para mostrar o progresso.


In [ ]:
# Calcula quantos filmes existem na base limpa.
total_rows = len(df_clean)

# Mostra quantos registros serão enviados ao Qdrant.
print(f"Total de filmes que serão vetorizados e salvos: {total_rows}")

# Cria um laço que anda pela tabela em lotes.
for start_index in tqdm(range(0, total_rows, BATCH_SIZE), desc="Enviando lotes ao Qdrant"):
    # Calcula o fim do lote atual.
    end_index = min(start_index + BATCH_SIZE, total_rows)

    # Seleciona as linhas do lote atual.
    batch_df = df_clean.iloc[start_index:end_index]

    # Pega os textos RAG do lote atual.
    batch_texts = batch_df["rag_text"].tolist()

    # Gera os embeddings do lote atual.
    batch_vectors = embedding_model.encode(
        # Envia a lista de textos para o modelo.
        batch_texts,

        # Mostra barra de progresso interna apenas se quiser.
        # Aqui deixamos False porque já temos a barra externa do tqdm.
        show_progress_bar=False,

        # Converte os vetores para numpy array.
        convert_to_numpy=True,

        # Normaliza os embeddings para melhorar comparação por cosseno.
        normalize_embeddings=True,
    )

    # Cria uma lista vazia para armazenar os pontos que serão enviados ao Qdrant.
    points = []

    # Percorre cada item do lote junto com seu embedding.
    for local_position, (_, row) in enumerate(batch_df.iterrows()):
        # Calcula o índice global do filme na tabela.
        global_index = start_index + local_position

        # Pega o vetor correspondente ao filme atual.
        vector = batch_vectors[local_position].tolist()

        # Monta o payload correspondente ao filme atual.
        payload = build_payload(row)

        # Cria o ponto do Qdrant com id, vetor e payload.
        point = models.PointStruct(
            # Define um id inteiro para o ponto.
            id=int(global_index),

            # Define o vetor do ponto.
            vector=vector,

            # Define os metadados do ponto.
            payload=payload,
        )

        # Adiciona o ponto na lista do lote.
        points.append(point)

    # Envia o lote de pontos para o Qdrant.
    qdrant_client.upsert(
        # Define a coleção de destino.
        collection_name=COLLECTION_NAME,

        # Define os pontos que serão inseridos ou atualizados.
        points=points,
    )

# Mostra mensagem final de sucesso.
print("Todos os filmes foram salvos no Qdrant com sucesso.")


## Célula 14 — Conferir quantos pontos foram salvos

Vamos verificar se o Qdrant recebeu os registros.


In [ ]:
# Pede ao Qdrant as informações da coleção.
collection_info = qdrant_client.get_collection(collection_name=COLLECTION_NAME)

# Mostra as informações da coleção no notebook.
print("Informações da coleção:")
print(collection_info)

# Mostra a quantidade de pontos salvos, quando essa informação estiver disponível.
print(f"Quantidade aproximada de pontos: {collection_info.points_count}")


## Célula 15 — Função de busca semântica no Qdrant

Essa função recebe uma pergunta e retorna os filmes mais parecidos.


In [ ]:
# Define uma função para buscar filmes parecidos com a pergunta.
def search_movies_in_qdrant(question, top_k=TOP_K):
    # Gera o embedding da pergunta do usuário.
    question_vector = embedding_model.encode(
        # Texto da pergunta.
        question,

        # Converte o resultado para numpy array.
        convert_to_numpy=True,

        # Normaliza o vetor para comparação por cosseno.
        normalize_embeddings=True,
    )

    # Converte o vetor de numpy array para lista Python.
    question_vector = question_vector.tolist()

    # Tenta usar a API mais nova do qdrant-client.
    try:
        # Consulta os pontos mais próximos no Qdrant.
        result = qdrant_client.query_points(
            # Nome da coleção.
            collection_name=COLLECTION_NAME,

            # Vetor da pergunta.
            query=question_vector,

            # Quantidade máxima de resultados.
            limit=top_k,

            # Pede para o Qdrant devolver também os payloads.
            with_payload=True,
        )

        # Extrai a lista de pontos retornados.
        points = result.points

    # Se a versão instalada do qdrant-client não tiver query_points, usa o método antigo.
    except AttributeError:
        # Consulta usando o método search, comum em versões anteriores.
        points = qdrant_client.search(
            # Nome da coleção.
            collection_name=COLLECTION_NAME,

            # Vetor da pergunta.
            query_vector=question_vector,

            # Quantidade máxima de resultados.
            limit=top_k,

            # Pede para devolver payloads.
            with_payload=True,
        )

    # Retorna os pontos encontrados.
    return points


# Define uma pergunta de teste para demonstrar a busca.
test_question = "Quais filmes falam sobre crime ou máfia?"

# Busca filmes parecidos com a pergunta de teste.
test_results = search_movies_in_qdrant(test_question, top_k=3)

# Mostra a pergunta de teste.
print(f"Pergunta de teste: {test_question}")

# Percorre os resultados encontrados.
for i, point in enumerate(test_results, start=1):
    # Pega o payload do ponto.
    payload = point.payload

    # Pega o score de similaridade.
    score = point.score

    # Mostra o ranking, score e título.
    print(f"\nResultado {i} | Score: {score:.4f}")

    # Mostra o título do filme.
    print(f"Título: {payload.get('title')}")

    # Mostra o gênero do filme.
    print(f"Gênero: {payload.get('genre')}")

    # Mostra a sinopse do filme.
    print(f"Sinopse: {payload.get('overview')}")


## Célula 16 — Montar o contexto recuperado

O contexto recuperado será enviado para a Maritaca junto com a pergunta.


In [ ]:
# Define uma função para transformar os resultados do Qdrant em texto de contexto.
def build_context_from_results(results):
    # Cria uma lista vazia para guardar cada bloco de contexto.
    context_blocks = []

    # Percorre cada resultado retornado pelo Qdrant.
    for i, point in enumerate(results, start=1):
        # Pega os metadados do resultado.
        payload = point.payload

        # Pega o texto RAG salvo no Qdrant.
        rag_text = payload.get("rag_text", "")

        # Pega o score de similaridade.
        score = point.score

        # Monta um bloco de texto com número, score e conteúdo.
        block = (
            f"[Documento {i} | Similaridade: {score:.4f}]\n"
            f"{rag_text}"
        )

        # Adiciona o bloco na lista de contextos.
        context_blocks.append(block)

    # Junta todos os blocos com separadores visuais.
    context = "\n\n---\n\n".join(context_blocks)

    # Retorna o contexto final.
    return context


# Cria um contexto de exemplo usando os resultados da pergunta de teste.
test_context = build_context_from_results(test_results)

# Esse * 80 só repete o caractere "=" 80 vezes.
# É igual a mandar imprimir:
# ================================================================================
# Mostra o contexto gerado.
print("Contexto que será enviado para a LLM:")
print("=" * 80)
print(test_context)
print("=" * 80)


## Célula 17 — Criar cliente da Maritaca

A Maritaca será chamada com o cliente compatível com OpenAI.


In [ ]:
# Cria o cliente da API usando a chave carregada do .env.
maritaca_client = OpenAI(
    # Define a chave da API da Maritaca.
    api_key=MARITACA_API_KEY,

    # Define a URL base da API da Maritaca.
    base_url=MARITACA_BASE_URL,
)

# Mostra que o cliente foi criado.
print("Cliente da Maritaca criado com sucesso.")


## Célula 18 — Função para chamar o modelo `sabiazinho-4`

Essa função envia pergunta + contexto para a LLM e recebe a resposta.


In [ ]:
# Define uma função para gerar resposta usando a Maritaca.
def ask_maritaca(question, context):
    # Monta uma mensagem de sistema explicando o papel do assistente.
    system_message = (
        "Você é um assistente de RAG para uma aula introdutória. "
        "Responda em português do Brasil. "
        "Use somente o contexto fornecido sobre filmes do IMDb. "
        "Se a resposta não estiver no contexto, diga que não encontrou informação suficiente. "
        "Seja claro, didático e breve."
    )

    # Monta a mensagem do usuário com contexto e pergunta.
    user_message = f"""
CONTEXTO RECUPERADO DO QDRANT:
{context}

PERGUNTA DO USUÁRIO:
{question}

INSTRUÇÃO:
Responda usando apenas o contexto recuperado.
"""

    # Envia a requisição para a API da Maritaca.
    response = maritaca_client.responses.create(
        # Define o modelo escolhido.
        model=MARITACA_MODEL_NAME,

        # Envia as mensagens no formato de conversa.
        input=[
            # Mensagem de sistema com regras gerais.
            {"role": "system", "content": system_message},

            # Mensagem do usuário com contexto e pergunta.
            {"role": "user", "content": user_message},
        ],

        # Define limite de tokens da resposta.
        max_output_tokens=500,

        # Usa temperatura baixa para respostas mais estáveis e menos aleatórias.
        temperature=0.2,
    )

    # Extrai o texto da resposta.
    answer = response.output[0].content[0].text

    # Retorna a resposta final.
    return answer


# Mostra uma mensagem antes do teste.
print("Função ask_maritaca criada.")


## Célula 19 — Teste completo do RAG

Agora juntamos:

1. Pergunta do usuário.
2. Embedding da pergunta.
3. Busca no Qdrant.
4. Contexto recuperado.
5. Resposta da Maritaca.


In [ ]:
# Define uma pergunta de exemplo para testar o fluxo completo.
demo_question = "Indique filmes de drama com boa avaliação e explique rapidamente."

# Mostra a pergunta escolhida.
print(f"Pergunta: {demo_question}")

# Busca documentos parecidos no Qdrant.
demo_results = search_movies_in_qdrant(demo_question, top_k=TOP_K)

# Monta o contexto com os documentos encontrados.
demo_context = build_context_from_results(demo_results)

# Mostra os títulos recuperados antes de chamar a LLM.
print("\nFilmes recuperados do Qdrant:")
for i, point in enumerate(demo_results, start=1):
    # Pega o payload do documento recuperado.
    payload = point.payload

    # Mostra título e score.
    print(f"{i}. {payload.get('title')} | Score: {point.score:.4f}")

# Chama a Maritaca para gerar a resposta final.
demo_answer = ask_maritaca(demo_question, demo_context)

# Mostra a resposta final no notebook.
print("\nResposta da Maritaca:")
print("=" * 80)
print(demo_answer)
print("=" * 80)


## Célula 20 — Função final do chatbot RAG

Essa função organiza tudo em uma única chamada.


In [ ]:
# Define uma função de alto nível para responder perguntas com RAG.
def answer_with_rag(question, top_k=TOP_K, show_sources=True):
    # Busca os documentos mais parecidos com a pergunta.
    results = search_movies_in_qdrant(question, top_k=top_k)

    # Monta o contexto textual a partir dos documentos recuperados.
    context = build_context_from_results(results)

    # Gera a resposta final usando a Maritaca.
    answer = ask_maritaca(question, context)

    # Se show_sources for True, mostra os documentos usados como fonte.
    if show_sources:
        # Mostra cabeçalho das fontes.
        print("\nFontes recuperadas do Qdrant:")

        # Percorre cada resultado recuperado.
        for i, point in enumerate(results, start=1):
            # Pega os metadados do documento.
            payload = point.payload

            # Mostra número, título e similaridade.
            print(f"{i}. {payload.get('title')} | Similaridade: {point.score:.4f}")

    # Retorna a resposta e os resultados para análise posterior.
    return answer, results


# Testa a função final com uma pergunta simples.
final_test_question = "Quais filmes parecem envolver aventura?"

# Mostra a pergunta de teste.
print(f"Pergunta de teste: {final_test_question}")

# Executa o RAG.
final_test_answer, final_test_sources = answer_with_rag(final_test_question)

# Mostra a resposta.
print("\nResposta:")
print(final_test_answer)


## Célula 21 — Loop de perguntas

Aqui o aluno pode digitar perguntas até escrever `sair`.


In [ ]:
# Mostra uma mensagem inicial para o usuário.
print("Chat RAG iniciado.")

# Explica como parar o loop.
print("Digite sua pergunta ou escreva 'sair' para encerrar.")

# Inicia um loop infinito.
while True:
    # Lê uma pergunta digitada pelo usuário.
    user_question = input("\nPergunta: ")

    # Remove espaços extras do começo e do fim da pergunta.
    user_question = user_question.strip()

    # Verifica se o usuário quer encerrar o programa.
    if user_question.lower() in ["sair", "exit", "quit", "parar"]:
        # Mostra uma mensagem de encerramento.
        print("Encerrando o chat RAG. Até mais!")

        # Para o loop.
        break

    # Verifica se o usuário digitou uma pergunta vazia.
    if user_question == "":
        # Pede para o usuário digitar algo.
        print("Digite uma pergunta válida.")

        # Volta para o início do loop.
        continue

    # Mostra que o sistema vai buscar informações.
    print("\nBuscando no Qdrant e consultando a Maritaca...")

    # Executa o RAG para responder a pergunta.
    answer, sources = answer_with_rag(user_question, top_k=TOP_K, show_sources=True)

    # Mostra a resposta final.
    print("\nResposta:")
    print("=" * 80)
    print(answer)
    print("=" * 80)


## Célula 22 — Opcional: apagar a coleção

Use esta célula somente se quiser limpar o Qdrant e refazer tudo.


In [ ]:
# Esta variável funciona como uma trava de segurança.
# Troque para True apenas se realmente quiser apagar a coleção.
APAGAR_COLECAO = False

# Verifica se a trava foi ativada.
if APAGAR_COLECAO:
    # Apaga a coleção do Qdrant.
    qdrant_client.delete_collection(collection_name=COLLECTION_NAME)

    # Mostra confirmação de exclusão.
    print(f"Coleção '{COLLECTION_NAME}' apagada.")
else:
    # Mostra que nada foi apagado.
    print("Nada foi apagado. Para apagar, mude APAGAR_COLECAO para True.")
